# Editing a model with ROME (Rank-One Model Editing) and detecting the edit

> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.** Check each tool's current documentation and `--help` before trusting a flag or an API name.

**Goal.** Understand how one fact inside a language model can be changed by a tiny weight update, why that is a
supply-chain risk (a "stealth" model), and how to detect it.

| | |
|---|---|
| Runtime | Part 1 and 2: CPU is fine (GPT-2 small). Part 3 (real ROME on GPT-2 XL): GPU runtime, about 16 GB GPU RAM. |
| Time | Part 1-2: 15 minutes. Part 3: 30-60 minutes, mostly installs and downloads. |
| Offline twin | `labs/rome-model-editing/` in this repo (a toy key-value memory, runs without a GPU). Do that first. |

**Safety.** This notebook edits a harmless fact ("The Eiffel Tower is in Paris" becomes "Rome"), the example from the ROME
paper. Do not upload edited models to a public hub pretending they are the original. That is what the 2023 *PoisonGPT*
demonstration by Mithril Security warned about: a model edited with ROME, published under a look-alike name.

## What ROME does, in simple words

Some layers of a transformer (the MLP layers) act like a key-value memory. The *key* is a hidden vector that stands for a
subject ("The Eiffel Tower"). The *value* is what the layer writes back, and it carries the fact ("is in Paris").
ROME finds a layer, computes a new value that makes the model say the new fact, and adds a **rank-one matrix** to that
layer's weight so that this key now produces the new value. Other keys are changed as little as possible.

Security view: the edit is tiny, touches one weight matrix, and leaves benchmark scores almost unchanged. A normal
accuracy test will not notice it. A hash of the weights will.

## Part 1. A manual rank-one patch on GPT-2 small (no ROME library needed)

In [ ]:
!pip install -q transformers==4.44.2

In [ ]:
import hashlib, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def state_hashes(m):
    # one SHA-256 per tensor, so we can see WHICH tensor changed
    out = {}
    for name, p in m.state_dict().items():
        out[name] = hashlib.sha256(p.detach().cpu().contiguous().numpy().tobytes()).hexdigest()
    return out

reference = state_hashes(model)           # imagine this came from a signed manifest
reference_weights = {k: v.clone() for k, v in model.state_dict().items() if k.endswith("mlp.c_proj.weight")}
print(len(reference), "tensors hashed")

In [ ]:
# A ROME-like change: add a small rank-one matrix u v^T to ONE MLP output matrix.
# This is NOT the full ROME algorithm (no causal tracing, no optimised value vector).
# It only shows what the *shape* of the change looks like: one tensor, rank one.
layer = 5
name = f"transformer.h.{layer}.mlp.c_proj.weight"
W = model.state_dict()[name]              # GPT-2 uses Conv1D: shape [3072, 768]
torch.manual_seed(0)
u = torch.randn(W.shape[0], 1); v = torch.randn(1, W.shape[1])
with torch.no_grad():
    W += 0.01 * (u @ v)
print("patched", name, tuple(W.shape))

### Detection A: hash comparison with the trusted reference

In [ ]:
now = state_hashes(model)
changed = [k for k in reference if reference[k] != now[k]]
print("tensors whose hash changed:", changed)

### Detection B: the weight difference is rank one (a fingerprint of this kind of edit)

In [ ]:
delta = (model.state_dict()[name] - reference_weights[name]).float()
print("rank of the weight difference:", torch.linalg.matrix_rank(delta).item())
print("size (Frobenius norm):", delta.norm().item())

### Detection C: behaviour tests (and why they are not enough)

A regression suite only catches an edit if it asks about the edited fact. Try a few prompts. Most outputs will look
normal. Real ROME edits are much more specific than the crude patch above.

In [ ]:
for prompt in ["The Eiffel Tower is located in the city of", "The capital of France is", "Water boils at"]:
    ids = tok(prompt, return_tensors="pt")
    out = model.generate(**ids, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    print(repr(tok.decode(out[0], skip_special_tokens=True)))

## Part 2. Questions to answer in your notes

1. Why is a hash check stronger than an accuracy check against a targeted edit?
2. What must you trust for the hash check to mean anything? (Hint: where does the reference hash come from?)
3. If the attacker also edits the manifest, what stops them? (Hint: signatures. See `notebooks/model-signing-cosign.ipynb`.)

## Part 3 (optional, GPU). The real ROME

Two public options. Both are research code and change often. **Read the project's README first.**

* **Original code** by the ROME authors (Meng et al.): the GitHub repository `kmeng01/rome`. It pins old versions of
  PyTorch and transformers, so it may not install on a current Colab image.
* **EasyEdit** (the `zjunlp/EasyEdit` project) wraps ROME and other editors with one API and is usually easier to run.

The cells below follow the EasyEdit style. The names (`BaseEditor`, `ROMEHyperParams`, the hyper-parameter file path, the
`edit` arguments) are written from memory: **check the EasyEdit README and your installed version.**

In [ ]:
# Heavy: this downloads gpt2-xl (about 6 GB) and needs a GPU. Run only if you have the time and the runtime.
!git clone -q https://github.com/zjunlp/EasyEdit.git
%cd EasyEdit
!pip install -q -r requirements.txt

In [ ]:
from easyeditor import BaseEditor, ROMEHyperParams

hparams = ROMEHyperParams.from_hparams("./hparams/ROME/gpt2-xl")   # path may differ in your version
editor = BaseEditor.from_hparams(hparams)

metrics, edited_model, _ = editor.edit(
    prompts=["The Eiffel Tower is located in the city of"],
    ground_truth=["Paris"],
    target_new=["Rome"],
    subject=["The Eiffel Tower"],
    keep_original_weight=False,
)
print(metrics)

In [ ]:
# Compare the edited model with a fresh copy: which tensors changed, and what is the rank of the change?
import torch
from transformers import AutoModelForCausalLM
orig = AutoModelForCausalLM.from_pretrained("gpt2-xl")
sd_o, sd_e = orig.state_dict(), edited_model.state_dict()
for k in sd_o:
    if not torch.equal(sd_o[k].cpu(), sd_e[k].cpu()):
        d = (sd_e[k].cpu() - sd_o[k].cpu()).float()
        print(k, "rank of change:", torch.linalg.matrix_rank(d).item())

## Wrap-up

* A targeted edit changes very little, but it is **not invisible**: hashes, diffs against a trusted copy and signed manifests catch it.
* Do not rely on benchmark scores to prove a downloaded model is the model you think it is.
* Download models from the publisher's verified account, pin the revision (commit hash), and verify signatures when they exist.